# Sorghum disease model evaluation plots

This notebook evaluates MobileNetV2, EfficientNetB0, ResNet50, the Scratch CNN, and their equal-weight score ensemble on the existing grouped test split. It produces comparison charts, class-level metrics, and count and row-normalized confusion matrices for all five outputs.

It reads saved model files and test images; it does not retrain or modify any models. Run in the project Python environment with TensorFlow, PyTorch, scikit-learn, Matplotlib, NumPy, and Pillow installed. Inference covers about 1,075 images. GPU is used where available.

These are dataset evaluation results, not confirmed diagnoses or a field-performance guarantee. The ensemble averages class-score vectors as in the project's evaluator, not the reported accuracy values.


## 1. Find the project and prepare libraries

Set `SORGHUM_PROJECT_DIR` if your project folder is in a different location.

In [ ]:
from pathlib import Path
import os, sys, json, csv
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

roots = [Path(os.environ.get("SORGHUM_PROJECT_DIR", ".")), Path("E:/ASA Self len project"), Path("/mnt/e/ASA Self len project"), Path.cwd()]
PROJECT = next((p.resolve() for p in roots if (p / "evaluate_grouped_candidates.py").is_file()), None)
if PROJECT is None:
    raise FileNotFoundError("Set SORGHUM_PROJECT_DIR to the Sorghum project folder.")
sys.path.insert(0, str(PROJECT))

import tensorflow as tf
import torch
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from scratch_cnn_model import SorghumCNN, prepare_scratch_image

ARTIFACTS = {
    "MobileNetV2": ("mobilenetv2_grouped_candidate.keras", "class_names_mobilenetv2_grouped.json"),
    "EfficientNetB0": ("efficientnetb0_grouped_candidate.keras", "class_names_efficientnetb0_grouped.json"),
    "ResNet50": ("resnet50_grouped_candidate.keras", "class_names_resnet50_grouped.json"),
    "Scratch CNN": ("scratch_sorghum_cnn_grouped_candidate.pt", "class_names_scratch_grouped.json"),
}
DATA_DIR = PROJECT / "dataset_split_grouped_v2"
MANIFEST = PROJECT / "dataset_split_grouped_v2_manifest.csv"
OUTPUT = Path(os.environ.get("SORGHUM_PLOT_OUTPUT", PROJECT / "model_plots"))
OUTPUT.mkdir(exist_ok=True)
print("Project:", PROJECT, "\nTensorFlow GPUs:", tf.config.list_physical_devices("GPU"), "\nPyTorch CUDA:", torch.cuda.is_available())


## 2. Load saved models and predict the held-out images

This follows the same preprocessing and class-score alignment used by `evaluate_grouped_candidates.py`.

In [ ]:
def names_from_file(filename):
    return json.loads((PROJECT / filename).read_text(encoding="utf-8"))

def align(scores, source, target):
    if set(source) != set(target):
        raise ValueError("Model class names do not match.")
    return scores[:, [source.index(label) for label in target]]

target_classes = names_from_file(ARTIFACTS["MobileNetV2"][1])
model_classes = {name: names_from_file(pair[1]) for name, pair in ARTIFACTS.items()}
mobile = tf.keras.models.load_model(PROJECT / ARTIFACTS["MobileNetV2"][0], compile=False)
efficient = tf.keras.models.load_model(PROJECT / ARTIFACTS["EfficientNetB0"][0], compile=False)
resnet = tf.keras.models.load_model(PROJECT / ARTIFACTS["ResNet50"][0], compile=False)

scratch_names = model_classes["Scratch CNN"]
checkpoint = torch.load(PROJECT / ARTIFACTS["Scratch CNN"][0], map_location="cpu", weights_only=True)
if checkpoint.get("classes") != scratch_names:
    raise ValueError("Scratch model class list does not match its label file.")
scratch = SorghumCNN(len(scratch_names))
scratch.load_state_dict(checkpoint["model_state"])
scratch.eval()
scratch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scratch = scratch.to(scratch_device)
scratch_size = int(checkpoint.get("image_size", 224))

test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", image_size=(224, 224), batch_size=32,
    label_mode="int", class_names=target_classes, shuffle=False,
)
paths = list(test_ds.file_paths)
with MANIFEST.open(newline="", encoding="utf-8") as f:
    manifest_paths = {r["split_path"].replace("\\", "/") for r in csv.DictReader(f) if r["split"] == "test"}
folder_paths = {Path(p).resolve().relative_to(PROJECT.resolve()).as_posix() for p in paths}
if folder_paths != manifest_paths:
    raise ValueError("Test folder does not match the grouped split manifest.")

true_parts, score_parts = [], {name: [] for name in ARTIFACTS}
offset = 0
for batch, labels in test_ds:
    raw = batch.numpy().astype(np.float32)
    predictions = {
        "MobileNetV2": mobile.predict(mobilenet_preprocess(raw.copy()), verbose=0),
        "EfficientNetB0": efficient.predict(raw, verbose=0),
        "ResNet50": resnet.predict(resnet_preprocess(raw.copy()), verbose=0),
    }
    scratch_inputs = []
    for file in paths[offset:offset + len(raw)]:
        with Image.open(file) as src:
            image = ImageOps.exif_transpose(src).convert("RGB")
        scratch_inputs.append(prepare_scratch_image(image, scratch_size))
    with torch.inference_mode():
        predictions["Scratch CNN"] = torch.softmax(scratch(torch.cat(scratch_inputs).to(scratch_device)), dim=1).cpu().numpy()
    for name, values in predictions.items():
        score_parts[name].append(align(values, model_classes[name], target_classes))
    true_parts.append(labels.numpy())
    offset += len(raw)

y_true = np.concatenate(true_parts).astype(int)
scores = {name: np.concatenate(parts) for name, parts in score_parts.items()}
scores["Equal-weight ensemble"] = np.mean(np.stack(list(scores.values())), axis=0)
y_pred = {name: values.argmax(axis=1) for name, values in scores.items()}
print(f"Generated predictions for {len(y_true):,} test images across {len(y_pred)} outputs.")


## 3. Calculate overall and class-level metrics

In [ ]:
reports = {name: classification_report(y_true, y_pred[name], labels=range(6), target_names=target_classes, output_dict=True, zero_division=0) for name in y_pred}
summary = {}
class_values = {}
for name, report in reports.items():
    summary[name] = {"Accuracy": accuracy_score(y_true, y_pred[name])}
    for average, prefix in (("macro avg", "Macro"), ("weighted avg", "Weighted")):
        for metric in ("precision", "recall", "f1-score"):
            summary[name][f"{prefix} {metric.replace('-score', '').title()}"] = report[average][metric]
    class_values[name] = {}
    for label in target_classes:
        r = report[label]
        class_values[name][label] = {"Precision": r["precision"], "Recall": r["recall"], "F1": r["f1-score"], "Support": int(r["support"])}

model_names = list(summary)
metric_columns = list(summary[model_names[0]])
print("Model metric summary (percent):")
print("Model                  " + "  ".join(f"{m:>17}" for m in metric_columns))
for name in model_names:
    print(f"{name:<22}" + "  ".join(f"{summary[name][m]:17.2%}" for m in metric_columns))
print("\nPer-class metrics:")
for name in model_names:
    print("\n", name)
    for label in target_classes:
        v = class_values[name][label]
        print(f"{label:<26} P {v['Precision']:.2%}  R {v['Recall']:.2%}  F1 {v['F1']:.2%}  n={v['Support']}")


## 4. Compare accuracy, precision, recall, and F1

The first chart shows accuracy and macro averages. The second shows support-weighted averages. Both y-axes start at 80%, so read the axis labels when comparing bar heights.

In [ ]:
macro = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1"]
weighted = ["Weighted Precision", "Weighted Recall", "Weighted F1"]

def comparison_plot(columns, title, palette):
    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(model_names))
    width = 0.82 / len(columns)
    for i, metric in enumerate(columns):
        values = [summary[name][metric] * 100 for name in model_names]
        bars = ax.bar(x + (i - (len(columns) - 1) / 2) * width, values, width=width, label=metric, color=palette[i])
        ax.bar_label(bars, fmt="%.1f", padding=2, fontsize=8)
    ax.set(title=title, ylabel="Score (%)", ylim=(80, 101), xticks=x, xticklabels=model_names)
    ax.grid(axis="y", alpha=.25); ax.tick_params(axis="x", rotation=15); ax.legend(loc="lower right", ncol=2)
    fig.tight_layout()
    return fig

for columns, title, palette in [
    (macro, "Accuracy and macro metrics", ["#2C6C96", "#4F9D69", "#E49B35", "#8057A5"]),
    (weighted, "Support-weighted metrics", ["#4384A8", "#69A77D", "#8A6BB2"]),
]:
    fig = comparison_plot(columns, title, palette); plt.show(); plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=True)
for ax, metric in zip(axes, ["Precision", "Recall", "F1"]):
    values = np.array([[class_values[model][label][metric] for model in model_names] for label in target_classes])
    im = ax.imshow(values * 100, cmap="YlGnBu", vmin=70, vmax=100, aspect="auto")
    ax.set_title(f"Per-class {metric}"); ax.set_xticks(range(len(model_names)), model_names, rotation=35, ha="right"); ax.set_yticks(range(6), target_classes)
    for row in range(6):
        for col in range(len(model_names)): ax.text(col, row, f"{values[row,col]*100:.1f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, label="Score (%)")
fig.suptitle("Per-class precision, recall, and F1 by model"); fig.tight_layout(); plt.show()


## 5. Confusion matrices for all five outputs

The first figure shows raw counts. The second normalizes each row by that true class's support.

In [ ]:
matrices = {name: confusion_matrix(y_true, y_pred[name], labels=range(6)) for name in y_pred}
short_labels = ["Anthracnose", "Grain molds", "Covered smut", "Head Smut", "Rust", "loose smut"]

def plot_confusions(normalize=False):
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, (name, matrix) in zip(axes.flat, matrices.items()):
        values = matrix.astype(float)
        if normalize:
            totals = values.sum(axis=1, keepdims=True)
            values = np.divide(values, totals, out=np.zeros_like(values), where=totals != 0) * 100
        im = ax.imshow(values, cmap="YlOrRd" if normalize else "Blues", vmin=0, vmax=100 if normalize else None)
        ax.set_title(name); ax.set_xticks(range(6), short_labels, rotation=45, ha="right"); ax.set_yticks(range(6), short_labels)
        ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
        for r in range(6):
            for c in range(6):
                label = f"{values[r,c]:.1f}%" if normalize else str(int(matrix[r,c]))
                ax.text(c, r, label, ha="center", va="center", fontsize=7)
        fig.colorbar(im, ax=ax, fraction=.046, pad=.04)
    axes.flat[-1].axis("off")
    fig.suptitle("Row-normalized confusion matrices (%)" if normalize else "Confusion matrices (test-set counts)")
    fig.tight_layout(); plt.show()

plot_confusions(normalize=False)
plot_confusions(normalize=True)


## 6. Save the figures and metric tables

Files are saved in `model_plots` inside the project folder by default. Set `SORGHUM_PLOT_OUTPUT` to save elsewhere.

In [ ]:
with (OUTPUT / "overall_model_metrics.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f); writer.writerow(["Model"] + metric_columns)
    for name in model_names: writer.writerow([name] + [f"{summary[name][m]:.8f}" for m in metric_columns])

with (OUTPUT / "per_class_model_metrics.csv").open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f); writer.writerow(["Model", "Class", "Precision", "Recall", "F1", "Support"])
    for name in model_names:
        for label in target_classes:
            v = class_values[name][label]
            writer.writerow([name, label, f"{v['Precision']:.8f}", f"{v['Recall']:.8f}", f"{v['F1']:.8f}", v["Support"]])

for columns, title, filename, palette in [
    (macro, "Accuracy and macro metrics", "accuracy_macro_metrics.png", ["#2C6C96", "#4F9D69", "#E49B35", "#8057A5"]),
    (weighted, "Support-weighted metrics", "weighted_metrics.png", ["#4384A8", "#69A77D", "#8A6BB2"]),
]:
    fig = comparison_plot(columns, title, palette); fig.savefig(OUTPUT / filename, dpi=180, bbox_inches="tight"); plt.close(fig)

fig, axes = plt.subplots(1, 3, figsize=(18, 7), sharey=True)
for ax, metric in zip(axes, ["Precision", "Recall", "F1"]):
    values = np.array([[class_values[model][label][metric] for model in model_names] for label in target_classes])
    im = ax.imshow(values * 100, cmap="YlGnBu", vmin=70, vmax=100, aspect="auto")
    ax.set_title(f"Per-class {metric}"); ax.set_xticks(range(len(model_names)), model_names, rotation=35, ha="right"); ax.set_yticks(range(6), target_classes)
    for row in range(6):
        for col in range(len(model_names)): ax.text(col, row, f"{values[row,col]*100:.1f}", ha="center", va="center", fontsize=8)
    fig.colorbar(im, ax=ax, label="Score (%)")
fig.suptitle("Per-class precision, recall, and F1 by model"); fig.tight_layout(); fig.savefig(OUTPUT / "per_class_metrics.png", dpi=180, bbox_inches="tight"); plt.close(fig)

for normalized, filename in [(False, "confusion_matrices_counts.png"), (True, "confusion_matrices_normalized.png")]:
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, (name, matrix) in zip(axes.flat, matrices.items()):
        values = matrix.astype(float)
        if normalized:
            totals = values.sum(axis=1, keepdims=True)
            values = np.divide(values, totals, out=np.zeros_like(values), where=totals != 0) * 100
        im = ax.imshow(values, cmap="YlOrRd" if normalized else "Blues", vmin=0, vmax=100 if normalized else None)
        ax.set_title(name); ax.set_xticks(range(6), short_labels, rotation=45, ha="right"); ax.set_yticks(range(6), short_labels)
        ax.set_xlabel("Predicted class"); ax.set_ylabel("True class")
        for r in range(6):
            for c in range(6): ax.text(c, r, f"{values[r,c]:.1f}%" if normalized else str(matrix[r,c]), ha="center", va="center", fontsize=7)
        fig.colorbar(im, ax=ax, fraction=.046, pad=.04)
    axes.flat[-1].axis("off"); fig.suptitle("Row-normalized confusion matrices (%)" if normalized else "Confusion matrices (test-set counts)")
    fig.tight_layout(); fig.savefig(OUTPUT / filename, dpi=180, bbox_inches="tight"); plt.close(fig)

print("Saved PNG plots and CSV tables to:", OUTPUT)


### Files created by the final cell

- `accuracy_macro_metrics.png`: Accuracy and macro Precision, Recall, F1.
- `weighted_metrics.png`: support-weighted Precision, Recall, F1.
- `per_class_metrics.png`: class-level Precision, Recall, and F1 for each model.
- `confusion_matrices_counts.png`: count matrices for all four classifiers and the ensemble.
- `confusion_matrices_normalized.png`: row-normalized versions of those matrices.
- `overall_model_metrics.csv` and `per_class_model_metrics.csv`: reusable metric tables.

The test images come from the same source collection used for model development; test-set results are not a field-accuracy guarantee.
